In [3]:
import os
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working directory:", Path.cwd())

PROJECT_DIR = Path.cwd().parent

print("Project directory:", PROJECT_DIR)
print(
    "Project exists:",
    (PROJECT_DIR / "data" / "raw" / "users.csv").exists()
)

Python: e:\ecommerce_data_platform\.venv\Scripts\python.exe
Working directory: e:\ecommerce_data_platform\notebooks
Project directory: e:\ecommerce_data_platform
Project exists: True


In [4]:
import csv

csv_path = PROJECT_DIR / "data" / "raw" / "users.csv"

with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
    reader = csv.reader(file)
    header = next(reader)
    row_count = sum(1 for _ in reader)

print("File size:", csv_path.stat().st_size, "bytes")
print("Column count:", len(header))
print("Row count:", row_count)
print("Header:", header)

File size: 20003418 bytes
Column count: 16
Row count: 100000
Header: ['ID', 'FIRST_NAME', 'LAST_NAME', 'EMAIL', 'AGE', 'GENDER', 'STATE', 'STREET_ADDRESS', 'POSTAL_CODE', 'CITY', 'COUNTRY', 'LATITUDE', 'LONGITUDE', 'TRAFFIC_SOURCE', 'CREATED_AT', 'USER_GEOM']


In [5]:
import os
from dotenv import load_dotenv
import snowflake.connector

load_dotenv(PROJECT_DIR / ".env")

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    authenticator="PROGRAMMATIC_ACCESS_TOKEN",
    token=os.environ["SNOWFLAKE_TOKEN"],
    role="ECOM_LOADER",
    warehouse="ECOM_WH",
    database="ECOM_RAW",
    schema="THELOOK",
    enable_server_session_keep_alive_auto_detection=True,
)

with conn.cursor() as cursor:
    cursor.execute("""
        SELECT
            CURRENT_USER(),
            CURRENT_ROLE(),
            CURRENT_WAREHOUSE(),
            CURRENT_DATABASE(),
            CURRENT_SCHEMA()
    """)
    print(cursor.fetchone())

('zhuchaocheng9', 'ECOM_LOADER', 'ECOM_WH', 'ECOM_RAW', 'THELOOK')


In [6]:
with conn.cursor() as cursor:
    cursor.execute(
        "LIST @ECOM_RAW.THELOOK.ECOM_LOAD_STAGE/users"
    )

    stage_files = cursor.fetchall()

    for file_info in stage_files:
        print(file_info)

('ecom_load_stage/users/users.csv.gz', 4706448, '69b2eee57e2c3e1982ff18dda3d195b4', 'Wed, 28 Oct 2026 14:29:07 GMT')


In [8]:
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT
            CURRENT_ROLE(),
            CURRENT_DATABASE(),
            CURRENT_SCHEMA()
    """)
    print(cursor.fetchone())

('ECOM_LOADER', 'ECOM_RAW', 'THELOOK')


In [11]:
with conn.cursor() as cursor:
    cursor.execute("""
        SHOW TABLES LIKE 'STG_USERS'
        IN SCHEMA ECOM_RAW.THELOOK
    """)
    for row in cursor.fetchall():
        print(row)

(datetime.datetime(2026, 10, 28, 7, 1, 55, 14000, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'STG_USERS', 'ECOM_RAW', 'THELOOK', 'TABLE', '', '', 100000, 5970432, 'ACCOUNTADMIN', '1', 'OFF', 'OFF', 'OFF', None, None, 'N', 'N', 'ROLE', 'N', 'N', 'N', 'N', 'N', 'N', 'OFF', 'OFF')


In [12]:
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT COUNT(*)
        FROM ECOM_RAW.THELOOK.STG_USERS
    """)

    print("STG_USERS rows:", cursor.fetchone()[0])

STG_USERS rows: 100000


In [13]:
with conn.cursor() as cursor:
    cursor.execute(
        "LIST @ECOM_RAW.THELOOK.ECOM_LOAD_STAGE/users"
    )

    stage_files = cursor.fetchall()

    for file_info in stage_files:
        print(file_info)

('ecom_load_stage/users/users.csv.gz', 4706448, '69b2eee57e2c3e1982ff18dda3d195b4', 'Wed, 28 Oct 2026 14:29:07 GMT')


In [14]:
with conn.cursor() as cursor:
    cursor.execute("""
        SELECT
            FILE_NAME,
            STATUS,
            ROW_COUNT,
            ERROR_COUNT,
            FIRST_ERROR_MESSAGE
        FROM TABLE(
            ECOM_RAW.INFORMATION_SCHEMA.COPY_HISTORY(
                TABLE_NAME => 'STG_USERS',
                START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())
            )
        )
        ORDER BY LAST_LOAD_TIME DESC
        LIMIT 10
    """)

    for row in cursor.fetchall():
        print(row)

('users/users.csv.gz', 'Loaded', 100000, 0, None)


In [15]:
csv_path = PROJECT_DIR / "data" / "raw" / "users.csv"

local_uri = "file://" + csv_path.resolve().as_posix()

with conn.cursor() as cursor:
    cursor.execute(
        f"PUT '{local_uri}' "
        "@ECOM_RAW.THELOOK.ECOM_LOAD_STAGE/debug "
        "AUTO_COMPRESS=TRUE OVERWRITE=TRUE"
    )

    for row in cursor.fetchall():
        print(row)

('users.csv', 'users.csv.gz', 20003418, 4706448, 'NONE', 'GZIP', 'UPLOADED', '')
